<a href="https://colab.research.google.com/github/Svein-Tore/colab/blob/main/FOPDT-binder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import FloatSlider, Button, VBox, HBox, Output, FileUpload
from IPython.display import display, Markdown
import io

# === 1. Oppsett for filopplasting ===
# accept='' tillater alle filtyper
uploader = FileUpload(accept='', multiple=False, description="Last opp fil")
main_output = Output()

def start_analysen(change):
    with main_output:
        main_output.clear_output()
        if not uploader.value:
            return

        # Henter fildata (støtter ulike versjoner av ipywidgets)
        try:
            # For ipywidgets 8.0+
            file_item = uploader.value[0]
        except:
            # For eldre versjoner
            file_item = list(uploader.value.values())[0]

        content = file_item['content']
        df = pd.read_csv(io.BytesIO(content), sep=None, engine='python', decimal=',')

        tid_data = df.iloc[:,0].values
        niva_data = df.iloc[:,1].values

        # === 2. Automatisk estimering (FOPDT) ===
        y0_est = niva_data[0]
        A_est = niva_data[-1] - y0_est

        # Terskelverdier for tidskonstanten
        thresh10 = y0_est + 0.1 * A_est
        thresh85 = y0_est + 0.85 * A_est
        thresh63 = y0_est + 0.63 * A_est

        # Finn indekser
        idx10 = np.where(niva_data > thresh10)[0]
        idx85 = np.where(niva_data > thresh85)[0]
        idx63 = np.where(niva_data > thresh63)[0]

        # Estimer parametre (med sikkerhet hvis data er ufullstendige)
        L_est10 = tid_data[idx10[0]] if len(idx10) > 0 else 0
        L_est85 = tid_data[idx85[0]] if len(idx85) > 0 else 1
        L_est = max(0, abs(L_est10 - 0.05 * (L_est85 - L_est10)))

        T_est = max(0.1, (tid_data[idx63[0]] - L_est)) if len(idx63) > 0 else 10

        # === 3. Plott-funksjon ===
        def plot_fopdt(A, T, L, y0, save_png=False):
            # Modellformel: y = y0 + A * (1 - exp(-(t-L)/T))
            y_model = np.where(tid_data < L, y0, y0 + A * (1 - np.exp(-(tid_data - L) / T)))

            fig, ax = plt.subplots(figsize=(10, 5))
            ax.plot(tid_data, niva_data, "b.", markersize=3, label="Måledata")
            ax.plot(tid_data, y_model, "r-", linewidth=2, label=f"Modell: Δy={A:.2f}, T={T:.2f}")

            ax.set_xlabel("Tid [s]")
            ax.set_ylabel("Nivå / Respons")
            ax.grid(True, which='both', linestyle='--', alpha=0.5)
            ax.legend()

            if save_png:
                plt.savefig("FOPDT_plot.png", dpi=300)
                print("Plott lagret som FOPDT_plot.png")
            plt.show()

        # === 4. Widgets for interaktivitet ===
        A_slider = FloatSlider(value=A_est, min=A_est*0.5, max=A_est*1.5, step=0.01, description="Δy")
        T_slider = FloatSlider(value=T_est, min=T_est*0.1, max=T_est*3, step=0.1, description="T")
        L_slider = FloatSlider(value=L_est, min=0, max=max(10, L_est*5), step=0.1, description="L")
        y0_slider = FloatSlider(value=y0_est, min=y0_est-2, max=y0_est+2, step=0.01, description="y0")
        save_btn = Button(description="Lagre PNG", button_style='success')

        plot_out = Output()

        def update_plot(change):
            with plot_out:
                plot_out.clear_output(wait=True)
                plot_fopdt(A_slider.value, T_slider.value, L_slider.value, y0_slider.value)

        for s in [A_slider, T_slider, L_slider, y0_slider]:
            s.observe(update_plot, "value")

        save_btn.on_click(lambda b: plot_fopdt(A_slider.value, T_slider.value, L_slider.value, y0_slider.value, True))

        # Vis resultater inne i main_output
        display(Markdown(f"**Autoestimat fra data:** $\Delta y$={A_est:.2f}, $T$={T_est:.2f}, $L$={L_est:.2f}, $y_0$={y0_est:.2f}"))
        display(VBox([plot_out, A_slider, T_slider, L_slider, y0_slider, save_btn]))
        update_plot(None)

# === 5. Instruksjonstekst (Ligger fast nederst) ===
instruksjoner = Markdown(r"""
---
## Instruksjoner til elevene
Dra sliderne for Δy, T, L og y0 for å tilpasse modellen til måledataene.

## Praktisk test av PID-regulator (IMC/SIMC)
Bruk tabellen under for å velge λ og beregne dine PID-parametre.


| Valg av λ | Forventet regulering | Hva du bør observere |
| :--- | :--- | :--- |
| λ = T/2 | Rolig | Lite oversving, tregere respons |
| λ = T/4 | Standard | God balanse mellom fart og stabilitet |
| λ = T/6 | Rask | Kort innreguleringstid, mulig oversving |
""")

# Koble sammen knappen og funksjonen
uploader.observe(start_analysen, names='value')

# Vis startskjermen
display(Markdown("# FOPDT Modell-tilpasning"))
display(Markdown("### 1. Last opp måledata (.csv eller .txt)"), uploader)
display(main_output)
display(instruksjoner)